# NSGA-II（geatpy）多目标优化：风险改善最大化 + 经济成本最小化（基于 GWR 原始输出经 robust sigmoid 映射后的风险概率）

本 Notebook 使用 **geatpy 的 NSGA-II**（`moea_NSGA2_templet`）对 **GWR 原始回归输出经训练阶段保存的 robust sigmoid 参数映射后的风险概率** 做 **双目标优化**。

## 两个目标函数
- **目标1：相对风险改善率（越高越好）**  
  记基线风险概率为 \(y_{base}\)，优化后风险概率为 \(y_{opt}\)，定义  
  \[
  RR = \frac{y_{base}-y_{opt}}{\max(y_{base}, \varepsilon)}
  \]  
  其中 \(\varepsilon\) 是很小的数，避免除零。`RR` 越大表示风险下降越多（改善越明显）。

> 这里的 `y_base` / `y_opt` 不是简单 `clip(0,1)` 的结果。  
> 它们统一使用 `gwr_classification_artifacts.pkl` 中保存的 `transform_metadata`，
> 对 GWR 原始输出做同一套 robust sigmoid 映射后得到。

- **目标2：总经济成本（越低越好）**  
  \[
  C_{total}=C_{ISA}+C_{WT}+C_{LAI}
  \]  
  三个成本函数按你给的公式与参数计算（见后续代码块），并累加成总成本。

## 决策变量（仅3个）
- `IP`：Impervious Surface Area / ImperviousIndex（0–100，按该行值上下浮动；只允许减少最多30%）
- `LAI`：Leaf Area Index（按该行值 ±30%）
- `WTD`：Water Table Depth（按该行值 ±30%）

其余 7 个特征保持为该行原值，不参与优化，但参与模型预测计算。

---

## 运行方式
仍然使用 **注释/取消注释**选择要优化的点（单点 / topN / 区间），然后运行 NSGA-II。最后会输出：
- `summary`：每个点选取一个“折中解（knee/ideal距离最小）”的结果
- `pareto`：每个点的帕累托前沿解集（长表）


In [ ]:
# =========================
# 1) 环境与路径设置（先运行）
# =========================
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle

import geatpy as ea

# --- 关键：mgtwr 是本地代码目录（你给的路径） ---
# Linux / macOS 路径（你当前的写法）：
sys.path.append("/path/to/sinkhole-risk-china/code")

# Windows 备用（如果你在 Windows 上跑，就用这个路径替换上面的）：
# sys.path.append(r"D:\path\to\sinkhole-risk-china\code") 

# 用于确认 mgtwr 是否能 import（可选）
from mgtwr.sel_bw import Sel_BW
from mgtwr.model import Model
import mgtwr.gwr_sigmoid_utils as gwr_sigmoid_utils

print("Imports OK. mgtwr and gwr_sigmoid_utils are available.")


In [ ]:
ssp = "ssp2"  # TODO: 根据需要手动改成 'hist' / 'ssp1' / 'ssp2' / 'ssp3' / 'ssp4' / 'ssp5'
ssp_time = "2080"  # TODO: 根据需要手动改成 '2020' / '2040' / '2060' / '2080' / '2100'

resolution = "10km"  # TODO: 根据需要把分辨率手动改成 '0.1km' / '1km' / '3km' / '5km' / '10km' / '20km'
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_base_path = os.path.join(base_path, "data")

path_name = f"Points_China_all_{resolution}"
print(path_name)
# ✅ 输出目录：base_path/outputs/path_name
out_dir = os.path.join(base_path, "outputs", path_name, ssp, ssp_time, "optimization")
print(f"out_dir:{out_dir}")
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# ==================================
# 2) 文件路径 & 运行参数（按需修改）
# ==================================

# 你的 CSV（包含 Longitude, Latitude 和 10 个特征列）
# CSV_PATH = os.path.join(data_base_path, "Extracted_HAVE_future", f"Points_China_all_{resolution}", f"AllFeatures_Points_China_all_{resolution}_{ssp}_cleaned_optimization.csv")
CSV_PATH = os.path.join(
    data_base_path,
    "Extracted_HAVE_future",
    f"Points_China_all_{resolution}",
    f"AllFeatures_Points_China_all_{resolution}_{ssp}_cleaned.csv"
)

# 你的模型 pkl（内部应包含 key='gwr'）
MODEL_PKL_PATH = os.path.join(base_path, "code", "3_gwr_model_train", "national", "GWR", "gwr_model_national_GWR.pkl")
CLASSIFICATION_ARTIFACTS_PATH = os.path.join(
    base_path,
    "code",
    "3_gwr_model_train",
    "national",
    "GWR",
    "gwr_classification_artifacts.pkl",
)
CLASS_BOUNDARIES_PATH = os.path.join(
    base_path,
    "code",
    "3_gwr_model_train",
    "national",
    "GWR",
    "class_boundaries.pkl",
)
# -------- NSGA-II 参数（可调）--------
# 速度优先预设：当前 GWR 预测代价很高，默认切到 fast，避免 60x80 的超大搜索规模。
# 可选："balanced" / "fast" / "ultra_fast"
SPEED_PRESET = "fast"

if SPEED_PRESET == "balanced":
    NIND = 36
    MAXGEN = 30
    PM = 0.18
    XOVER = 0.75
    RANDOM_SAMPLE_FRAC_DEFAULT = 0.10
    SAVE_EVERY_ROWS_DEFAULT = 50
elif SPEED_PRESET == "ultra_fast":
    NIND = 16
    MAXGEN = 10
    PM = 0.12
    XOVER = 0.70
    RANDOM_SAMPLE_FRAC_DEFAULT = 0.10
    SAVE_EVERY_ROWS_DEFAULT = 150
else:
    SPEED_PRESET = "fast"
    NIND = 24
    MAXGEN = 16
    PM = 0.15
    XOVER = 0.72
    RANDOM_SAMPLE_FRAC_DEFAULT = 0.10
    SAVE_EVERY_ROWS_DEFAULT = 100

SEED = 1    # 基础随机种子（每个 row 会自动用 SEED + row_id，便于断点续跑复现）

print(f"Config set. SPEED_PRESET = {SPEED_PRESET}")
print(f"Estimated search load per row ~= NIND x MAXGEN = {NIND * MAXGEN}")
print("Random sample ratio is fixed at 10% by requirement.")

# ==============================
# 成本模型设置（2040-2060，默认20年累计）
# ==============================
T_YEARS = 20.0          # 2040-2060：20年

def infer_cell_area_km2(resolution_str: str) -> float:
    """根据 resolution（如 10km / 5km / 0.1km）自动推算网格面积 km^2。"""
    s = str(resolution_str).strip().lower()
    if not s.endswith("km"):
        raise ValueError(f"无法从 resolution={resolution_str!r} 自动识别网格边长，请手动设置 A_CELL_KM2。")
    side_km = float(s[:-2])
    return side_km ** 2

# 关键：自动从 resolution 推断网格面积，避免 10km 数据却仍沿用 1000 km^2
A_CELL_KM2 = infer_cell_area_km2(resolution)   # 例如 10km -> 100 km^2
A_M2 = float(A_CELL_KM2) * 1e6
A_HA = A_M2 / 1e4
print(f"A_CELL_KM2 (auto-inferred from resolution={resolution}): {A_CELL_KM2} km^2")

# 成本输出口径
COST_MODE = "incremental"   # "absolute" 或 "incremental"
COST_REPORT = "total"       # "total"（20年累计）或 "annual_avg"（年均=total/T_YEARS）
RISK_SPACE = "sigmoid_prob"

# -------- 成本平衡参数（把三类治理成本控制在同一数量级）--------
# bal11: 使用 bal9 的搜索空间保持 2/3级降低能力，但改用更均衡的最终挑解评分。
#        目标是在接近 bal9 等级效果的同时，把三类成本均值比压低。
BALANCE_TAG = "bal11"
IP_MAX_REDUCTION_FRAC = 1.00
LAI_REL_CHANGE_FRAC = 0.80
WTD_REL_CHANGE_FRAC = 1.00
ISA_NET_COST_CAP_USD = 2.0e8
LAI_NET_COST_CAP_USD = 2.5e7
WTD_NET_COST_CAP_USD = 5.0e7

# 内部优化权重：不改单价，只通过目标函数软权重降低 ISA 被系统性压制的问题，
# 同时略微抬高 LAI 的惩罚，避免三类成本继续大幅失衡。
INTERNAL_COST_WEIGHT_ISA = 0.03
INTERNAL_COST_WEIGHT_WT = 0.50
INTERNAL_COST_WEIGHT_LAI = 1.40
INTERNAL_BALANCE_PENALTY_WEIGHT = 3.00
RISK_LEVEL_DROP_OBJECTIVE_WEIGHT = 4.00
RISK_LEVEL_2PLUS_OBJECTIVE_BONUS = 2.00
RISK_LEVEL_3PLUS_OBJECTIVE_BONUS = 3.00

# 最终从帕累托解中挑选折中解时，再加入一层“分项成本不要差太多”的偏好。
KNEE_RR_WEIGHT = 1.00
KNEE_COST_WEIGHT = 0.50
KNEE_DROP_WEIGHT = 6.00
KNEE_BALANCE_LOG_WEIGHT = 2.00
KNEE_BALANCE_SHARE_WEIGHT = 4.00
KNEE_ISA_SHARE_WEIGHT = 0.00
KNEE_LAI_SHARE_EXCESS_WEIGHT = 0.00
KNEE_CAP_EXCESS_WEIGHT = 100.0
KNEE_MIN_RR_FRACTION = 0.00

# -------- 随机抽样（为了减少 9 万多点的计算量）--------
RANDOM_SAMPLE_ENABLED = True
RANDOM_SAMPLE_FRAC = RANDOM_SAMPLE_FRAC_DEFAULT
RANDOM_SAMPLE_SEED = 42
if RANDOM_SAMPLE_ENABLED:
    if not (0.0 < float(RANDOM_SAMPLE_FRAC) <= 1.0):
        raise ValueError("RANDOM_SAMPLE_FRAC 必须在 (0, 1] 范围内。")
    SAMPLE_TAG = f"sample{int(round(float(RANDOM_SAMPLE_FRAC) * 100)):02d}pct_seed{int(RANDOM_SAMPLE_SEED)}"
else:
    SAMPLE_TAG = "sample100pct"
SAMPLE_COST_WEIGHT = 1.0  # 实际值会在读入并抽样后按 n_full / n_sample 更新

# -------- 断点续跑 / 中间结果保存（强烈建议开启）--------
# 注意：
# - checkpoint / final 文件名现在只保留必要信息：种群规模、进化代数、成本口径、抽样标签、平衡版本。
# - 当前这版成本平衡逻辑与旧 checkpoint 不兼容，因此单独使用新的 BALANCE_TAG 避免混用。
# - 若你明确想从头重跑，请把 RESUME_FROM_CHECKPOINT 改为 False，或手动删除对应 checkpoint 文件。
# - SAVE_EVERY_ROWS 建议不要太小，否则会导致频繁硬盘写入。
RESUME_FROM_CHECKPOINT = True
SAVE_EVERY_ROWS = SAVE_EVERY_ROWS_DEFAULT
FSYNC_EVERY_SAVE = False

mode_tag = "inc" if str(COST_MODE).lower() == "incremental" else "abs"
report_tag = "tot" if str(COST_REPORT).lower() == "total" else "avg"
sample_tag = SAMPLE_TAG.replace("sample", "s").replace("pct_seed", "_r")
run_tag = f"{SPEED_PRESET}_n{NIND}_g{MAXGEN}_{mode_tag}_{report_tag}_{sample_tag}_{BALANCE_TAG}"
run_tag = run_tag.replace("/", "_").replace(" ", "_")

CHECKPOINT_SUMMARY_PATH = os.path.join(out_dir, f"nsga2_{run_tag}_summary_ckpt.csv")
CHECKPOINT_PARETO_PATH  = os.path.join(out_dir, f"nsga2_{run_tag}_pareto_ckpt.csv")
FINAL_SUMMARY_PATH = os.path.join(out_dir, f"nsga2_{run_tag}_summary.csv")
FINAL_PARETO_PATH = os.path.join(out_dir, f"nsga2_{run_tag}_pareto.csv")

# ---- 成本参数（来自你给的表格）----
# ISA
C_DEPAVE = 60.0     # USD/m^2 （一次性）
C_FEE = 2.0         # USD/m^2/year （年度）
# WTD
C_ENERGY = 0.01     # USD/m^3·m
C_CAPITAL = 0.0005  # USD/m^3·m^2
# LAI
C_PLANT = 2328.0    # USD/ha （一次性）
C_MAINT = 100.0     # USD/ha/year （年度）

print(f"Run tag = {run_tag}")
print(f"Checkpoint summary: {CHECKPOINT_SUMMARY_PATH}")
print(f"Checkpoint pareto : {CHECKPOINT_PARETO_PATH}")
print(f"Risk space = {RISK_SPACE}")
print(f"Random sample enabled = {RANDOM_SAMPLE_ENABLED}")
print(f"Random sample frac = {RANDOM_SAMPLE_FRAC}")
print(f"Random sample seed = {RANDOM_SAMPLE_SEED}")
print(f"Checkpoint save every rows = {SAVE_EVERY_ROWS}")
print(f"Fsync every save = {FSYNC_EVERY_SAVE}")
print(f"Classification artifacts: {CLASSIFICATION_ARTIFACTS_PATH}")
print(f"Class boundaries: {CLASS_BOUNDARIES_PATH}")
print(f"Cost caps (20y total): ISA={ISA_NET_COST_CAP_USD:,.0f}, LAI={LAI_NET_COST_CAP_USD:,.0f}, WTD={WTD_NET_COST_CAP_USD:,.0f}")
print(f"Relative bounds: IP -{IP_MAX_REDUCTION_FRAC:.0%}, LAI ±{LAI_REL_CHANGE_FRAC:.0%}, WTD ±{WTD_REL_CHANGE_FRAC:.0%}")
ISA_NET_COST_PER_POINT_USD = max(float(C_DEPAVE) - float(T_YEARS) * float(C_FEE), 0.0) * (float(A_M2) / 100.0)
print(f"ISA net cost per 1 percentage-point reduction = {ISA_NET_COST_PER_POINT_USD:,.0f} USD")
print(
    f"Internal objective weights: ISA={INTERNAL_COST_WEIGHT_ISA:.2f}, "
    f"WT={INTERNAL_COST_WEIGHT_WT:.2f}, LAI={INTERNAL_COST_WEIGHT_LAI:.2f}, "
    f"balance_penalty={INTERNAL_BALANCE_PENALTY_WEIGHT:.2f}"
)
print(
    f"Knee-selection weights: RR={KNEE_RR_WEIGHT:.2f}, "
    f"Cost={KNEE_COST_WEIGHT:.2f}, Drop={KNEE_DROP_WEIGHT:.2f}, LogBalance={KNEE_BALANCE_LOG_WEIGHT:.2f}, "
    f"ShareBalance={KNEE_BALANCE_SHARE_WEIGHT:.2f}, CapExcess={KNEE_CAP_EXCESS_WEIGHT:.1f}, "
    f"ISAShare={KNEE_ISA_SHARE_WEIGHT:.2f}, LAIShareExcess={KNEE_LAI_SHARE_EXCESS_WEIGHT:.2f}, "
    f"MinRRFraction={KNEE_MIN_RR_FRACTION:.2f}"
)

# Debug mode: set to True to shrink dataset by a factor of 1000 for fast testing
DEBUG_MODE = False


In [ ]:
# ==============================================
# 3) 列名定义（严格对齐预测 notebook 的特征顺序）
# ==============================================
FEATURE_COLS = [
    "Distance_to_karst",
    "Depth_to_Bedrock",
    "Distance_to_Fault_m",
    "UrbanFrac_2040_2060",
    "PopTotal_2040_2060",
    "ImperviousIndex_2040_2060",
    "LAI_2040_2060",
    "Precip_2040_2060",
    "WTD_2040_2060",
    "HDS_2040_2060",
    "Tas_2040_2060",
    "Huss_2040_2060",
]
SYMBOLS = ["DK", "DB", "DF", "UF", "PT", "IP", "LAI", "PR", "WTD", "HDS", "Tas", "Huss"]
IDX = {s: i for i, s in enumerate(SYMBOLS)}
N_FEATURES = len(FEATURE_COLS)

if len(FEATURE_COLS) != len(SYMBOLS):
    raise ValueError(
        f"FEATURE_COLS 长度({len(FEATURE_COLS)}) 与 SYMBOLS 长度({len(SYMBOLS)}) 不一致，请检查新增变量。"
    )

print("Feature columns:", FEATURE_COLS)
print("SYMBOLS:", SYMBOLS)
print("N_FEATURES:", N_FEATURES)


In [ ]:
# ==================================
# 4) 读入 CSV、随机抽样并检查列
# ==================================
df_full = pd.read_csv(CSV_PATH)

need_cols = ["Longitude", "Latitude", "NAME_EN_JX"] + FEATURE_COLS
missing = [c for c in need_cols if c not in df_full.columns]
if missing:
    raise KeyError(f"CSV 缺少必要列: {missing}")

df_full = df_full.reset_index().rename(columns={"index": "_source_row_id"})
n_full = int(df_full.shape[0])

if RANDOM_SAMPLE_ENABLED and float(RANDOM_SAMPLE_FRAC) < 1.0:
    sample_n = max(1, int(np.floor(n_full * float(RANDOM_SAMPLE_FRAC))))
    df = (
        df_full.sample(n=sample_n, random_state=int(RANDOM_SAMPLE_SEED), replace=False)
        .sort_values("_source_row_id")
        .reset_index(drop=True)
    )
else:
    sample_n = n_full
    df = df_full.copy().reset_index(drop=True)

SAMPLE_COST_WEIGHT = float(n_full) / float(sample_n)
df["_sample_weight"] = float(SAMPLE_COST_WEIGHT)

X0 = df[FEATURE_COLS].to_numpy(dtype=float)
print("Loaded full CSV:", df_full.shape)
print("Sampled df shape:", df.shape)
print(f"Actual sample ratio: {sample_n}/{n_full} = {sample_n / n_full:.4f}")
print(f"Sample cost weight (n_full / n_sample): {SAMPLE_COST_WEIGHT:.6f}")
print("X0 shape:", X0.shape)
print("NAME_EN_JX sample:", df["NAME_EN_JX"].head().tolist())


In [ ]:
# ==========================================================
# 5) 构造 coords：EPSG:4326 -> EPSG:3857，并拼成 (x, y, 0)
# ==========================================================
def build_coords(df: pd.DataFrame) -> np.ndarray:
    """
    优先走 geopandas + shapely（与你 notebook 常见写法一致），
    若环境没有 geopandas，则 fallback 到 pyproj。
    输出 coords: (n,3) = (x, y, 0)
    """
    if not {"Longitude", "Latitude"}.issubset(df.columns):
        raise KeyError("CSV 缺少 Longitude / Latitude 列。")

    lon = df["Longitude"].to_numpy(dtype=float)
    lat = df["Latitude"].to_numpy(dtype=float)

    # 1) geopandas 路径
    try:
        import geopandas as gpd
        from shapely.geometry import Point

        geometry = [Point(xy) for xy in zip(lon, lat)]
        gdf = gpd.GeoDataFrame(df.copy(), geometry=geometry, crs="EPSG:4326").to_crs("EPSG:3857")
        xy = np.array([(p.x, p.y) for p in gdf.geometry], dtype=float)

    except Exception:
        # 2) fallback：pyproj
        try:
            from pyproj import Transformer
            transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
            x, y = transformer.transform(lon, lat)
            xy = np.column_stack([x, y]).astype(float)
        except Exception as e:
            raise RuntimeError(
                "无法构造投影坐标。请安装 geopandas（推荐）或 pyproj。"
            ) from e

    coords = np.hstack([xy, np.zeros((xy.shape[0], 1), dtype=float)])
    return coords

coords = build_coords(df)
print("coords shape:", coords.shape)
print("coords sample:", coords[:3])

In [ ]:
# ===========================================
# 6) 载入模型 pkl，并提取 gwr 对象 + sigmoid transform_metadata
# ===========================================
import sys
import numpy.core as npcore
import numpy.core.numeric as np_numeric
import numpy.core.multiarray as np_multiarray

sys.modules.setdefault("numpy._core", npcore)
sys.modules.setdefault("numpy._core.numeric", np_numeric)
sys.modules.setdefault("numpy._core.multiarray", np_multiarray)

if not os.path.exists(MODEL_PKL_PATH):
    raise FileNotFoundError(f"GWR model pkl not found: {MODEL_PKL_PATH}")
if not os.path.exists(CLASSIFICATION_ARTIFACTS_PATH):
    raise FileNotFoundError(f"Classification artifacts pkl not found: {CLASSIFICATION_ARTIFACTS_PATH}")
if not os.path.exists(CLASS_BOUNDARIES_PATH):
    raise FileNotFoundError(f"Class boundaries pkl not found: {CLASS_BOUNDARIES_PATH}")

with open(MODEL_PKL_PATH, "rb") as f:
    saved = pickle.load(f)

if "gwr" not in saved:
    raise KeyError("模型 pkl 中找不到 key='gwr'。请确认保存结构与 notebook 一致。")

gwr = saved["gwr"]

with open(CLASSIFICATION_ARTIFACTS_PATH, "rb") as f:
    gwr_classification_artifacts = pickle.load(f)

with open(CLASS_BOUNDARIES_PATH, "rb") as f:
    class_boundary_data = pickle.load(f)

transform_metadata = gwr_classification_artifacts.get("transform_metadata")
if transform_metadata is None:
    raise KeyError("gwr_classification_artifacts.pkl 中缺少 transform_metadata，无法执行统一的 sigmoid 概率映射。")
class_breaks = np.sort(np.asarray(class_boundary_data.get("class_breaks", []), dtype=float).reshape(-1))
if class_breaks.size < 2 or (not np.isfinite(class_breaks).all()):
    raise KeyError("class_boundaries.pkl 中缺少有效的 class_breaks，无法执行等级降低优化。")

print("Loaded gwr model:", type(gwr))
print("Loaded transform_metadata:", transform_metadata)
print("Loaded class_breaks:", class_breaks)


In [ ]:
# ===========================================
# 7) 先计算所有点的原始 y_pred_gwr，并映射成 sigmoid 风险概率（用于排序/对比）
# ===========================================

import os
import contextlib
import numpy as np

def gwr_predict_batch_raw(gwr, coords: np.ndarray, X: np.ndarray) -> np.ndarray:
    """批量预测原始 GWR 分数：coords=(n,3), X=(n,n_features) -> raw_scores=(n,)"""
    # mgtwr/gwr 有时会往 stderr 打 tqdm，这里重定向 stderr 让输出更干净
    with open(os.devnull, "w") as fnull, contextlib.redirect_stderr(fnull):
        y_pred, _ = gwr.predict(coords, X)
    return np.asarray(y_pred, dtype=float).reshape(-1)


def gwr_scores_to_probability(raw_scores: np.ndarray, transform_metadata: dict) -> np.ndarray:
    """使用训练阶段保存的 robust sigmoid 参数，把原始 GWR 分数映射到 0-1 风险概率。"""
    probabilities, _ = gwr_sigmoid_utils.gwr_scores_to_probabilities(
        raw_scores,
        transform_metadata=transform_metadata,
    )
    return np.asarray(probabilities, dtype=float).reshape(-1)


def gwr_predict_batch(gwr, coords: np.ndarray, X: np.ndarray, transform_metadata: dict) -> np.ndarray:
    """批量预测并映射到 0-1 风险概率。"""
    raw_scores = gwr_predict_batch_raw(gwr, coords, X)
    probabilities = gwr_scores_to_probability(raw_scores, transform_metadata)
    return probabilities.reshape(-1, 1)


y0_raw = gwr_predict_batch_raw(gwr, coords, X0)
y0 = gwr_scores_to_probability(y0_raw, transform_metadata).reshape(-1, 1)
df["_y_pred_base_raw"] = y0_raw
df["_y_pred_base"] = y0.ravel()

print("Base raw GWR-score summary:")
print(pd.Series(y0_raw, name="_y_pred_base_raw").describe())
print("\nBase susceptibility-probability summary (robust sigmoid with saved transform_metadata):")
print(df["_y_pred_base"].describe())


In [ ]:
# =======================================================
# 8) 选择你要优化的点：单点 / topN / 区间（注释选择）
# =======================================================

# # 方式C：优化指定行号区间 [START, END)
START, END = 0, 49999   # 按需改
# START, END = 0, 10   # 按需改
indices = list(range(START, min(END, len(df))))
# if DEBUG_MODE:
#     indices = indices[:len(indices)//1000]
    
print("Rows to optimize:", len(indices))
print("First 10 indices:", indices[:10])

In [ ]:
# ======================================================
# 9) 定义上下限：固定变量不参与优化；仅优化 IP / LAI / WTD（按当前行 base 比例）
# ======================================================
# 关键点：
#   - 固定变量不作为决策变量参与优化
#   - 仅优化 3 个变量：IP(=ISA), LAI, WTD
#   - 使用“真实相对变化范围 + 分项预算 cap”双重约束，避免 WTD/LAI 成本爆炸

OPT_SYMBOLS = ["IP", "LAI", "WTD"]  # 决策变量（3维）
OPT_IDX10 = [IDX[s] for s in OPT_SYMBOLS]  # 在全部特征中的列索引

# ---- 物理/取值域（用于剪裁，防止出现不合理负值）----
ISA_MIN, ISA_MAX = 0.0, 100.0
LAI_MIN, LAI_MAX = 0.0, 543.0
# WTD 数据中存在负值；不能把下界硬剪到 0，否则负基线点会被错误推到 0 并产生巨额 WT 成本。
WTD_MIN, WTD_MAX = -4000.0, 4000.0

def _safe_minmax(a: float, b: float):
    """避免 base 可能为负时上下限反转。"""
    return (a, b) if a <= b else (b, a)


def _lai_scale_from_base(base_lai_raw: float) -> float:
    """LAI 原始值大于 20 时，按 ×100 存储处理。"""
    return 100.0 if float(base_lai_raw) > 20.0 else 1.0


def _wtd_delta_cap_from_budget(cost_cap_usd: float) -> float:
    """根据 WTD 成本函数反推在预算 cap 下允许的最大绝对变化量 d。"""
    cost_cap_usd = float(cost_cap_usd)
    if cost_cap_usd <= 0:
        return 0.0

    a = float(C_CAPITAL)
    b = float(C_ENERGY)
    c = -cost_cap_usd / max(float(A_M2), 1e-12)

    if a <= 0:
        if b <= 0:
            return np.inf
        return max(0.0, -c / b)

    disc = b * b - 4.0 * a * c
    disc = max(disc, 0.0)
    return max(0.0, (-b + np.sqrt(disc)) / (2.0 * a))


def _lai_raw_increase_cap_from_budget(base_lai_raw: float, cost_cap_usd: float) -> float:
    """根据 LAI 成本函数反推在预算 cap 下允许的最大原始值增量。"""
    cost_cap_usd = float(cost_cap_usd)
    if cost_cap_usd <= 0:
        return 0.0

    scale = _lai_scale_from_base(base_lai_raw)
    unit_cost_per_actual_lai = float(A_HA) * (float(C_PLANT) + float(T_YEARS) * float(C_MAINT))
    if unit_cost_per_actual_lai <= 0:
        return np.inf

    delta_actual_lai = cost_cap_usd / unit_cost_per_actual_lai
    return max(0.0, delta_actual_lai * scale)


def bounds_from_base(base10: np.ndarray):
    """根据当前行全维特征 base10，返回 3 个可控变量（IP,LAI,WTD）的 lb/ub。"""
    base10 = np.asarray(base10, dtype=float).reshape(-1)
    if base10.shape[0] != N_FEATURES:
        raise ValueError(f"base10 必须是长度为 {N_FEATURES} 的一维数组。当前长度={base10.shape[0]}")

    lb3 = np.zeros(3, dtype=float)
    ub3 = np.zeros(3, dtype=float)

    # ---- IP / ISA：只允许减少；同时受相对变化与预算 cap 双重约束 ----
    ip0 = float(np.clip(base10[IDX["IP"]], ISA_MIN, ISA_MAX))
    rel_cap_pts = float(IP_MAX_REDUCTION_FRAC) * ip0

    net_unit_usd_per_m2 = float(C_DEPAVE) - float(T_YEARS) * float(C_FEE)
    if net_unit_usd_per_m2 <= 0:
        budget_cap_pts = rel_cap_pts
    else:
        net_cost_per_point = net_unit_usd_per_m2 * (float(A_M2) / 100.0)
        budget_cap_pts = float(ISA_NET_COST_CAP_USD) / (net_cost_per_point + 1e-12)

    max_reduce_pts = float(np.clip(min(rel_cap_pts, budget_cap_pts), 0.0, ip0))
    lo = float(np.clip(ip0 - max_reduce_pts, ISA_MIN, ISA_MAX))
    hi = float(np.clip(ip0, ISA_MIN, ISA_MAX))
    lb3[0], ub3[0] = (lo, hi) if lo <= hi else (hi, lo)

    # ---- LAI：真实 ±30%，并限制“增加量”对应的最大成本 ----
    lai0 = float(np.clip(base10[IDX["LAI"]], LAI_MIN, LAI_MAX))
    lai_rel_lo, lai_rel_hi = _safe_minmax(
        lai0 * (1.0 - float(LAI_REL_CHANGE_FRAC)),
        lai0 * (1.0 + float(LAI_REL_CHANGE_FRAC)),
    )
    lai_up_cap = lai0 + _lai_raw_increase_cap_from_budget(lai0, LAI_NET_COST_CAP_USD)
    lo = float(np.clip(lai_rel_lo, LAI_MIN, LAI_MAX))
    hi = float(np.clip(min(lai_rel_hi, lai_up_cap), LAI_MIN, LAI_MAX))
    if hi < lo:
        hi = lo
    lb3[1], ub3[1] = lo, hi

    # ---- WTD：真实 ±30%，并限制绝对变化量对应的最大成本 ----
    wtd0 = float(base10[IDX["WTD"]])
    wtd_rel_lo, wtd_rel_hi = _safe_minmax(
        wtd0 * (1.0 - float(WTD_REL_CHANGE_FRAC)),
        wtd0 * (1.0 + float(WTD_REL_CHANGE_FRAC)),
    )
    wtd_delta_cap = _wtd_delta_cap_from_budget(WTD_NET_COST_CAP_USD)
    lo = max(wtd_rel_lo, wtd0 - wtd_delta_cap)
    hi = min(wtd_rel_hi, wtd0 + wtd_delta_cap)
    lo = float(np.clip(lo, WTD_MIN, WTD_MAX))
    hi = float(np.clip(hi, WTD_MIN, WTD_MAX))
    if hi < lo:
        hi = lo
    lb3[2], ub3[2] = lo, hi

    return lb3.tolist(), ub3.tolist()

# 快速检查一个点的上下限（可选）
test_i = indices[0] if len(indices) else 0
base10 = X0[test_i]
lb3_test, ub3_test = bounds_from_base(base10)

print("Decision vars bounds (based on THIS row base value):")
for j, s in enumerate(OPT_SYMBOLS):
    b = base10[IDX[s]]
    print(f"{s:>3}: lb={lb3_test[j]:.6g}, ub={ub3_test[j]:.6g}, base={b:.6g}")

print(f"WTD delta cap from budget: {_wtd_delta_cap_from_budget(WTD_NET_COST_CAP_USD):.6g} m")
print(f"LAI raw increase cap from budget: {_lai_raw_increase_cap_from_budget(base10[IDX['LAI']], LAI_NET_COST_CAP_USD):.6g}")


In [ ]:
# ======================================================
# 10) NSGA-II：单点优化函数（双目标：RR 最大化 + 成本最小化）
# ======================================================
# 目标1 RR：风险概率相对改善率（越大越好）：RR = (y_base - y_opt)/max(y_base, eps)
# 目标2 Cost：总经济成本（越小越好）：C_total = C_ISA + C_WT + C_LAI

# ---- 成本参数 & 时空尺度 ----
# 注意：这些参数默认在“2) 文件路径 & 运行参数”里统一设置（便于你集中改）
# - T_YEARS: 2040-2060 年数（默认 20）
# - A_CELL_KM2: 单元面积（默认 1000 km^2；如是 10km×10km 网格应为 100 km^2）
# - COST_MODE: 'absolute' or 'incremental'
# - COST_REPORT: 'total' or 'annual_avg'

A_M2 = float(A_CELL_KM2) * 1e6     # m^2
A_HA = A_M2 / 1e4                 # ha

EPS = 1e-6

import contextlib

def gwr_predict_singlecoord_raw(gwr, coord: np.ndarray, X: np.ndarray) -> np.ndarray:
    """单点坐标 + 多组X：coord=(3,), X=(N,n_features) -> raw_scores=(N,)"""
    coord = np.asarray(coord, dtype=float).reshape(1, -1)
    coord_rep = np.tile(coord, (X.shape[0], 1))
    with open(os.devnull, "w") as fnull, contextlib.redirect_stderr(fnull):
        y_pred, _ = gwr.predict(coord_rep, X)
    return np.asarray(y_pred, dtype=float).reshape(-1)


def gwr_predict_singlecoord(gwr, coord: np.ndarray, X: np.ndarray, transform_metadata: dict) -> np.ndarray:
    """单点坐标 + 多组X，返回映射后的 0-1 风险概率。"""
    raw_scores = gwr_predict_singlecoord_raw(gwr, coord, X)
    probabilities = gwr_scores_to_probability(raw_scores, transform_metadata)
    return probabilities.reshape(-1, 1)


def assemble_X10_from_X3(base10: np.ndarray, X3: np.ndarray) -> np.ndarray:
    """把 3 维决策变量（IP,LAI,WTD）拼回全维特征用于 GWR 预测。"""
    base10 = np.asarray(base10, dtype=float).reshape(-1)
    if base10.shape[0] != N_FEATURES:
        raise ValueError(f"base10 必须是长度为 {N_FEATURES} 的一维数组。当前长度={base10.shape[0]}")
    base10 = base10.reshape(1, N_FEATURES)
    X3 = np.asarray(X3, dtype=float)
    if X3.ndim == 1:
        X3 = X3.reshape(1, 3)
    if X3.shape[1] != 3:
        raise ValueError("X3 必须是 (N,3) 形状，对应 [IP, LAI, WTD].")

    X10 = np.repeat(base10, repeats=X3.shape[0], axis=0)
    X10[:, IDX["IP"]]  = X3[:, 0]
    X10[:, IDX["LAI"]] = X3[:, 1]
    X10[:, IDX["WTD"]] = X3[:, 2]
    return X10


def _lai_scale_from_base(base_lai_raw: float) -> float:
    """你的数据 LAI 范围到 400+，很可能是 ×100；这里自动做一个尺度判别。"""
    return 100.0 if base_lai_raw > 20.0 else 1.0


def cost_components(base10: np.ndarray, X3: np.ndarray):
    """
    按你给的成本函数计算 (C_ISA, C_WT, C_LAI, C_total)，返回形状 (N,1) 的数组。

    COST_MODE:
      - 'absolute'   : 输出情景下“总成本”（包含存量 ISA/LAI 的年度费用/维护）
      - 'incremental': 输出相对基线(base)的“治理增量成本”（更接近治理投入/预算口径；推荐）
        * ISA：depave 一次性成本 + (fee_opt - fee_base) 20年差额
        * LAI：plant 一次性成本 + (maint_opt - maint_base) 20年差额
        * WT ：本来就是基于变化量 d 的成本（增量口径）

    COST_REPORT:
      - 'total'      : 20年累计（与灾害时间尺度一致）
      - 'annual_avg' : 年均（= total / T_YEARS），便于直观对比“每年花费”
    """
    base10 = np.asarray(base10, dtype=float).reshape(-1)
    X3 = np.asarray(X3, dtype=float)
    if X3.ndim == 1:
        X3 = X3.reshape(1, 3)

    isa_orig = float(base10[IDX["IP"]])
    wt_orig  = float(base10[IDX["WTD"]])
    lai_orig_raw = float(base10[IDX["LAI"]])

    isa_opt = X3[:, 0]
    lai_opt_raw = X3[:, 1]
    wt_opt  = X3[:, 2]

    # ---- ISA 成本（IP 是 0-100 的比例）----
    isa_orig_c = float(np.clip(isa_orig, 0.0, 100.0))
    isa_opt_c  = np.clip(isa_opt,  0.0, 100.0)

    # 去硬化一次性成本：只对“减少的 ISA 面积”计费
    depave_pts = np.maximum(0.0, isa_orig_c - isa_opt_c)  # 单位：百分点
    C_ISA_depave = A_M2 * float(C_DEPAVE) * (depave_pts / 100.0)

    # 年度 fee：与 ISA 面积成正比
    fee_base = A_M2 * float(T_YEARS) * float(C_FEE) * (isa_orig_c / 100.0)
    fee_opt  = A_M2 * float(T_YEARS) * float(C_FEE) * (isa_opt_c  / 100.0)

    if str(COST_MODE).lower() == "incremental":
        # 增量口径：只看相对基线的差额（减少 ISA 会带来负成本=节省）
        C_ISA_fee = fee_opt - fee_base
    else:
        # 绝对口径：直接用情景下的总 fee
        C_ISA_fee = fee_opt

    C_ISA = C_ISA_depave + C_ISA_fee

    # ---- WT 成本（按变化量 d）----
    d = np.abs(wt_opt - wt_orig)
    C_WT = A_M2 * (float(C_ENERGY) * d + float(C_CAPITAL) * d**2)

    # ---- LAI 成本（自动判断是否需要 /100）----
    scale = _lai_scale_from_base(lai_orig_raw)
    lai_orig = max(lai_orig_raw / scale, 0.0)
    lai_opt  = np.maximum(lai_opt_raw / scale, 0.0)

    # 种植成本：只对 LAI 增加计一次性成本
    delta_lai_plus = np.maximum(0.0, lai_opt - lai_orig)
    C_LAI_plant = A_HA * float(C_PLANT) * delta_lai_plus

    # 维护成本：与 LAI 水平成正比（年度）
    maint_base = A_HA * float(T_YEARS) * float(C_MAINT) * lai_orig
    maint_opt  = A_HA * float(T_YEARS) * float(C_MAINT) * lai_opt

    if str(COST_MODE).lower() == "incremental":
        C_LAI_maint = maint_opt - maint_base
    else:
        C_LAI_maint = maint_opt

    C_LAI = C_LAI_plant + C_LAI_maint
    # ---- 强制成本非负（非常重要：目标函数中的 cost 必须 >= 0）----
    # 说明：在增量口径下，某些差额项可能为负（代表“节省”），但作为“花费”目标值不应为负。
    # 因此对各分项与总成本都做截断：cost = max(cost, 0)
    C_ISA = np.maximum(C_ISA, 0.0)
    C_WT  = np.maximum(C_WT,  0.0)
    C_LAI = np.maximum(C_LAI, 0.0)

    C_total = C_ISA + C_WT + C_LAI
    # 报告口径：年均 or 20年累计
    if str(COST_REPORT).lower() == "annual_avg":
        C_ISA = C_ISA / float(T_YEARS)
        C_WT  = C_WT  / float(T_YEARS)
        C_LAI = C_LAI / float(T_YEARS)
        C_total = C_total / float(T_YEARS)

    return (np.asarray(C_ISA).reshape(-1,1),
            np.asarray(C_WT).reshape(-1,1),
            np.asarray(C_LAI).reshape(-1,1),
            np.asarray(C_total).reshape(-1,1))


def component_cap_excess_ratio(cost_parts: np.ndarray) -> np.ndarray:
    """分项成本超过设定 cap 的相对超额比例；为 0 表示完全满足预算约束。"""
    cost_parts = np.maximum(np.asarray(cost_parts, dtype=float), 0.0)
    if cost_parts.ndim == 1:
        cost_parts = cost_parts.reshape(1, -1)
    caps = np.array([
        max(float(ISA_NET_COST_CAP_USD), EPS),
        max(float(WTD_NET_COST_CAP_USD), EPS),
        max(float(LAI_NET_COST_CAP_USD), EPS),
    ], dtype=float).reshape(1, 3)
    excess = np.maximum(cost_parts - caps, 0.0) / caps
    return np.sum(excess, axis=1)


def optimization_cost_objective(C_ISA: np.ndarray, C_WT: np.ndarray, C_LAI: np.ndarray) -> np.ndarray:
    """
    内部优化目标：不改单价，只通过软权重与不平衡惩罚，
    避免 ISA 长期被“每 1pct 约 2000 万 USD”的 10km 网格成本尺度压到几乎不参与。
    同时对超出分项预算 cap 的候选解施加很强的硬惩罚。
    """
    C_ISA = np.asarray(C_ISA, dtype=float).reshape(-1, 1)
    C_WT  = np.asarray(C_WT, dtype=float).reshape(-1, 1)
    C_LAI = np.asarray(C_LAI, dtype=float).reshape(-1, 1)

    weighted_total = (
        float(INTERNAL_COST_WEIGHT_ISA) * C_ISA +
        float(INTERNAL_COST_WEIGHT_WT)  * C_WT +
        float(INTERNAL_COST_WEIGHT_LAI) * C_LAI
    )

    cost_stack = np.hstack([C_ISA, C_WT, C_LAI])
    cost_mean = np.mean(cost_stack, axis=1, keepdims=True)
    imbalance = np.sqrt(np.mean((cost_stack - cost_mean) ** 2, axis=1, keepdims=True))
    cap_excess = component_cap_excess_ratio(cost_stack).reshape(-1, 1)
    hard_penalty = 1.0e9 * cap_excess

    return weighted_total + float(INTERNAL_BALANCE_PENALTY_WEIGHT) * imbalance + hard_penalty


def probability_to_risk_level(probabilities: np.ndarray) -> np.ndarray:
    """用训练阶段保存的自然断点把 0-1 概率映射为 1-5 级风险。"""
    probs = np.asarray(probabilities, dtype=float).reshape(-1)
    breaks = np.asarray(class_breaks, dtype=float).reshape(-1)
    probs = np.clip(probs, float(breaks[0]), float(breaks[-1]))
    return np.digitize(probs, breaks[1:-1], right=True) + 1


def risk_metrics(base_y: float, y_opt: np.ndarray):
    """返回 RR、等级降低数 drop，以及带等级奖励的优化效用 utility。"""
    y_opt = np.asarray(y_opt, dtype=float).reshape(-1)
    denom = max(float(base_y), EPS)
    RR = np.clip((float(base_y) - y_opt) / denom, 0.0, 1.0)

    base_level = int(probability_to_risk_level(np.array([base_y]))[0])
    opt_level = probability_to_risk_level(y_opt)
    drop = np.clip(base_level - opt_level, 0, class_breaks.size - 2).astype(float)

    drop_norm = drop / max(float(class_breaks.size - 2), 1.0)
    utility = (
        RR
        + float(RISK_LEVEL_DROP_OBJECTIVE_WEIGHT) * drop_norm
        + float(RISK_LEVEL_2PLUS_OBJECTIVE_BONUS) * (drop >= 2.0).astype(float)
        + float(RISK_LEVEL_3PLUS_OBJECTIVE_BONUS) * (drop >= 3.0).astype(float)
    )
    return RR.reshape(-1, 1), drop.reshape(-1, 1), utility.reshape(-1, 1)


def objectives(base_y: float, y_opt: np.ndarray, C_obj: np.ndarray):
    """构造两目标 ObjV = [Level-aware utility, Cost]。
    utility 同时包含连续 RR 和自然断点等级降低奖励，直接推动 2/3 级下降。
    """
    _, _, utility = risk_metrics(base_y, y_opt)
    return np.hstack([utility, np.asarray(C_obj).reshape(-1,1)])


def component_log_balance_penalty(cost_parts: np.ndarray) -> np.ndarray:
    """衡量三类成本是否处在相近量级；越小越平衡。"""
    cost_parts = np.maximum(np.asarray(cost_parts, dtype=float), 0.0)
    if cost_parts.ndim == 1:
        cost_parts = cost_parts.reshape(1, -1)
    log_costs = np.log1p(cost_parts)
    return np.max(log_costs, axis=1) - np.min(log_costs, axis=1)


def component_share_balance_penalty(cost_parts: np.ndarray) -> np.ndarray:
    """衡量三类成本占比是否接近 1/3 : 1/3 : 1/3；越小越平衡。"""
    cost_parts = np.maximum(np.asarray(cost_parts, dtype=float), 0.0)
    if cost_parts.ndim == 1:
        cost_parts = cost_parts.reshape(1, -1)
    total = np.sum(cost_parts, axis=1, keepdims=True)
    shares = np.divide(cost_parts, total, out=np.zeros_like(cost_parts), where=total > EPS)
    return np.mean((shares - (1.0 / 3.0)) ** 2, axis=1)


def pick_knee_solution(ObjV: np.ndarray, cost_parts=None):
    """
    在帕累托解中选一个折中解：
    1) 先保留 RR 至少达到该点 Pareto 最大 RR 的 KNEE_MIN_RR_FRACTION 的候选；
    2) 在高 RR 候选中，优先选择更高等级降低数，再惩罚总成本、量级不平衡、占比不平衡和超 cap；
    3) ObjV 第3/4列若存在，分别视为 utility / drop，用于 bal8 的等级感知挑解。
    """
    rr = np.asarray(ObjV[:, 0], dtype=float)
    cost = np.asarray(ObjV[:, 1], dtype=float)
    idx_all = np.arange(rr.shape[0])

    rr_max_all = float(np.nanmax(rr)) if rr.size else 0.0
    min_rr_fraction = float(globals().get("KNEE_MIN_RR_FRACTION", 0.0))
    if rr_max_all > EPS and min_rr_fraction > 0.0:
        candidate_mask = rr >= (min_rr_fraction * rr_max_all)
        if not np.any(candidate_mask):
            candidate_mask = np.ones_like(rr, dtype=bool)
    else:
        candidate_mask = np.ones_like(rr, dtype=bool)

    drop = np.asarray(ObjV[:, 3], dtype=float) if ObjV.shape[1] >= 4 else np.zeros_like(rr)
    rr_c = rr[candidate_mask]
    cost_c = cost[candidate_mask]
    drop_c = drop[candidate_mask]
    idx_c = idx_all[candidate_mask]

    rr_min, rr_max = float(np.min(rr_c)), float(np.max(rr_c))
    c_min, c_max   = float(np.min(cost_c)), float(np.max(cost_c))

    rr_norm = (rr_c - rr_min) / (rr_max - rr_min + EPS)
    c_norm  = (cost_c - c_min) / (c_max - c_min + EPS)

    drop_max = float(np.max(drop_c)) if drop_c.size else 0.0
    drop_norm = drop_c / max(drop_max, 1.0)
    score = (
        float(KNEE_RR_WEIGHT) * (1.0 - rr_norm) ** 2
        + float(KNEE_COST_WEIGHT) * (c_norm) ** 2
        - float(KNEE_DROP_WEIGHT) * drop_norm
    )

    if cost_parts is not None:
        cost_parts_c = np.asarray(cost_parts, dtype=float)[candidate_mask]
        log_bal = component_log_balance_penalty(cost_parts_c)
        share_bal = component_share_balance_penalty(cost_parts_c)
        cap_excess = component_cap_excess_ratio(cost_parts_c)
        total_parts = np.sum(np.maximum(cost_parts_c, 0.0), axis=1, keepdims=True)
        cost_shares = np.divide(
            np.maximum(cost_parts_c, 0.0),
            total_parts,
            out=np.zeros_like(cost_parts_c, dtype=float),
            where=total_parts > EPS,
        )
        isa_share = cost_shares[:, 0]
        lai_share = cost_shares[:, 2]

        lb_min, lb_max = float(np.min(log_bal)), float(np.max(log_bal))
        if lb_max > lb_min:
            lb_norm = (log_bal - lb_min) / (lb_max - lb_min + EPS)
            score = score + float(KNEE_BALANCE_LOG_WEIGHT) * (lb_norm ** 2)

        sb_min, sb_max = float(np.min(share_bal)), float(np.max(share_bal))
        if sb_max > sb_min:
            sb_norm = (share_bal - sb_min) / (sb_max - sb_min + EPS)
            score = score + float(KNEE_BALANCE_SHARE_WEIGHT) * (sb_norm ** 2)

        score = score + float(KNEE_CAP_EXCESS_WEIGHT) * cap_excess
        score = score - float(KNEE_ISA_SHARE_WEIGHT) * isa_share
        score = score + float(KNEE_LAI_SHARE_EXCESS_WEIGHT) * np.maximum(lai_share - 0.45, 0.0) ** 2

    return int(idx_c[int(np.argmin(score))])

def optimize_one_point(
    gwr,
    coord: np.ndarray,
    base10: np.ndarray,
    base_y: float,
    transform_metadata: dict,
    nind: int = 60,
    maxgen: int = 80,
    seed: int = 1,
    pm: float = 0.2,
    xover: float = 0.7,
):
    """
    对单个点做 NSGA-II 双目标优化：
      - 目标1：RR 最大化（风险改善越大越好）
      - 目标2：Cost 最小化（内部为平衡后的软成本目标）
    决策维度=3（IP,LAI,WTD），其余固定变量保持 base 值，但仍参与预测。

    返回：
        best_var10 (n_features,),
        best_RR (float),
        best_Cost (float),
        Vars3 (k,3),
        ObjV_actual (k,2)     # [RR, actual_total_cost]
    """
    base10 = np.asarray(base10, dtype=float).reshape(-1)
    lb3, ub3 = bounds_from_base(base10)

    class GWRProblem(ea.Problem):
        def __init__(self):
            name = "GWR_RR_Max_Cost_Min"
            M = 2
            maxormins = [-1, 1]     # RR 最大化；Cost 最小化
            Dim = 3
            varTypes = [0] * Dim
            lbin = [1] * Dim
            ubin = [1] * Dim
            ea.Problem.__init__(self, name, M, maxormins, Dim, varTypes, lb3, ub3, lbin, ubin)

        def evalVars(self, X3):
            X10 = assemble_X10_from_X3(base10, X3)
            y_opt = gwr_predict_singlecoord(gwr, coord, X10, transform_metadata)
            C_ISA, C_WT, C_LAI, _ = cost_components(base10, X3)
            C_obj = optimization_cost_objective(C_ISA, C_WT, C_LAI)
            ObjV = objectives(base_y, y_opt, C_obj)
            return ObjV

    problem = GWRProblem()
    pop = ea.Population(Encoding="RI", NIND=nind)
    algorithm = ea.moea_NSGA2_templet(problem, pop, MAXGEN=maxgen, logTras=0)

    algorithm.mutOper.Pm = pm
    algorithm.recOper.XOVR = xover

    res = ea.optimize(
        algorithm,
        seed=seed,
        verbose=False,
        drawing=0,
        outputMsg=False,
        saveFlag=False,
        drawLog=False,
    )

    Vars3 = res["Vars"]
    ObjV_internal = res["ObjV"]

    X10_all = assemble_X10_from_X3(base10, Vars3)
    y_opt_all = gwr_predict_singlecoord(gwr, coord, X10_all, transform_metadata).reshape(-1)
    RR_all, drop_all, utility_all = risk_metrics(base_y, y_opt_all)
    C_ISA_all, C_WT_all, C_LAI_all, C_total_all = cost_components(base10, Vars3)
    cost_parts = np.hstack([C_ISA_all, C_WT_all, C_LAI_all])
    # ObjV_actual: [真实RR, 真实总成本, 等级感知utility, 等级降低数drop]
    ObjV_actual = np.hstack([RR_all, C_total_all, utility_all, drop_all])

    best_i = pick_knee_solution(ObjV_actual, cost_parts)
    best_var3 = Vars3[best_i, :]
    best_var10 = assemble_X10_from_X3(base10, best_var3).reshape(-1)

    best_RR = float(ObjV_actual[best_i, 0])
    best_Cost = float(np.maximum(C_total_all[best_i, 0], 0.0))

    return best_var10, best_RR, best_Cost, Vars3, ObjV_actual

print("Balanced multi-objective optimization function ready.")


In [ ]:
# ======================================================
# 11) 执行优化（逐点）
#     输出：summary（每点一个折中解） + pareto（每点帕累托解集长表）
#     新增：断点续跑 + 批量落盘，降低频繁硬盘读写
# ======================================================
import gc
import time

FIXED_SYMBOLS = ["DK", "DB", "DF", "UF", "PT", "PR", "HDS", "Tas", "Huss"]  # 9个固定变量，顺序与预测 notebook 对齐
FIXED_IDX10 = [IDX[s] for s in FIXED_SYMBOLS]

def _read_ckpt_csv(path: str) -> pd.DataFrame:
    if (not os.path.exists(path)) or os.path.getsize(path) == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path, on_bad_lines="skip")
    except TypeError:
        # 老版本 pandas 没有 on_bad_lines
        return pd.read_csv(path)
    except Exception as e:
        print(f"Warning: failed to read checkpoint {path}: {e}")
        return pd.DataFrame()

def _append_df_safely(df_part: pd.DataFrame, path: str):
    """以 append 方式写 CSV。默认只 flush，不每次 fsync，避免频繁硬盘同步。"""
    if df_part is None or len(df_part) == 0:
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    need_header = (not os.path.exists(path)) or (os.path.getsize(path) == 0)
    with open(path, "a", encoding="utf-8-sig", newline="") as f:
        df_part.to_csv(f, index=False, header=need_header)
        f.flush()
        if FSYNC_EVERY_SAVE:
            os.fsync(f.fileno())

# ---- 载入已有 checkpoint，实现断点续跑 ----
summary_ckpt_df = _read_ckpt_csv(CHECKPOINT_SUMMARY_PATH)
pareto_ckpt_df  = _read_ckpt_csv(CHECKPOINT_PARETO_PATH)

if "row_id" in summary_ckpt_df.columns:
    summary_ckpt_df = summary_ckpt_df.drop_duplicates(subset=["row_id"], keep="last")
    done_set = set(summary_ckpt_df["row_id"].astype(int).tolist())
else:
    done_set = set()

if ("row_id" in pareto_ckpt_df.columns) and ("sol_id" in pareto_ckpt_df.columns):
    pareto_ckpt_df = pareto_ckpt_df.drop_duplicates(subset=["row_id", "sol_id"], keep="last")

if RESUME_FROM_CHECKPOINT:
    todo_indices = [int(i) for i in indices if int(i) not in done_set]
else:
    todo_indices = [int(i) for i in indices]

print(f"Checkpoint summary: {CHECKPOINT_SUMMARY_PATH}")
print(f"Checkpoint pareto : {CHECKPOINT_PARETO_PATH}")
print(f"Already finished : {len(done_set)} rows")
print(f"Rows to run now  : {len(todo_indices)} / {len(indices)}")

summary_buffer = []
pareto_buffer = []

for run_k, i in enumerate(todo_indices, start=1):
    base10 = X0[i, :]
    coord = coords[i, :]
    name_en_jx = str(df.loc[i, "NAME_EN_JX"])
    base_y = float(df.loc[i, "_y_pred_base"])   # 已在第7块映射到 [0,1] 风险概率

    # 跳过 NaN / inf
    if np.any(~np.isfinite(base10)) or np.any(~np.isfinite(coord)) or (not np.isfinite(base_y)):
        summary_row = {
            "row_id": i,
            "source_row_id": int(df.loc[i, "_source_row_id"]),
            "Longitude": float(df.loc[i, "Longitude"]),
            "Latitude": float(df.loc[i, "Latitude"]),
            "NAME_EN_JX": name_en_jx,
            "y_pred_base": base_y,
            "RR_best": np.nan,
            "Cost_best": np.nan,
            "y_pred_opt_best": np.nan,
            "sample_weight": float(df.loc[i, "_sample_weight"]),
            "status": "skip_nan",
        }
        summary_buffer.append(summary_row)

        if len(summary_buffer) >= int(SAVE_EVERY_ROWS):
            _append_df_safely(pd.DataFrame(summary_buffer), CHECKPOINT_SUMMARY_PATH)
            summary_buffer = []
        continue

    # 关键：每个 row 用固定且可复现的 seed，便于断点续跑后结果稳定
    row_seed = int(SEED) + int(i)

    best_var10, best_RR, best_Cost, Vars3, ObjV = optimize_one_point(
        gwr=gwr,
        coord=coord,
        base10=base10,
        base_y=base_y,
        transform_metadata=transform_metadata,
        nind=NIND,
        maxgen=MAXGEN,
        seed=row_seed,
        pm=PM,
        xover=XOVER,
    )

    # 再算一遍帕累托解对应的真实风险概率预测（批量、已通过 sigmoid 映射到[0,1]），避免 RR=0 时由公式反推失真
    X10_pareto = assemble_X10_from_X3(base10, Vars3)
    y_opt_all = gwr_predict_singlecoord(gwr, coord, X10_pareto, transform_metadata).reshape(-1)

    # 成本分项一次性向量化计算，避免在 summary / pareto 导出阶段重复逐条重算
    C_ISA_all, C_WT_all, C_LAI_all, C_total_all = cost_components(base10, Vars3)
    cost_parts = np.hstack([C_ISA_all, C_WT_all, C_LAI_all])

    # 与 optimize_one_point 内部一致：再找一次折中解索引
    best_i = pick_knee_solution(ObjV, cost_parts)
    y_opt_best = float(y_opt_all[best_i])   # 已是 [0,1] 风险概率

    # 用“真实且已映射”的风险概率重新计算 RR，强制范围 [0,1]
    denom = max(base_y, EPS)
    best_RR = float(np.clip((base_y - y_opt_best) / denom, 0.0, 1.0))
    best_Cost = float(np.maximum(C_total_all[best_i, 0], 0.0))

    # 固定变量自检（理论应为0）
    fixed_max_abs_diff = float(np.max(np.abs(best_var10[FIXED_IDX10] - base10[FIXED_IDX10])))

    # 折中解的 3 个可控变量
    IP_best  = float(best_var10[IDX["IP"]])
    LAI_best = float(best_var10[IDX["LAI"]])
    WTD_best = float(best_var10[IDX["WTD"]])

    # 折中解的成本分项
    C_ISA = float(C_ISA_all[best_i, 0])
    C_WT = float(C_WT_all[best_i, 0])
    C_LAI = float(C_LAI_all[best_i, 0])
    C_total = float(C_total_all[best_i, 0])

    summary_row = {
        "row_id": i,
        "source_row_id": int(df.loc[i, "_source_row_id"]),
        "Longitude": float(df.loc[i, "Longitude"]),
        "Latitude": float(df.loc[i, "Latitude"]),
        "NAME_EN_JX": name_en_jx,
        "y_pred_base": float(np.clip(base_y, 0.0, 1.0)),
        "y_pred_opt_best": float(np.clip(y_opt_best, 0.0, 1.0)),
        "RR_best": best_RR,                  # 已强制 [0,1]
        "Cost_best": best_Cost,              # 已强制 >=0
        "Cost_ISA": max(C_ISA, 0.0),
        "Cost_WT": max(C_WT, 0.0),
        "Cost_LAI": max(C_LAI, 0.0),
        "Cost_total_check": max(C_total, 0.0),

        "IP_opt": IP_best,
        "LAI_opt": LAI_best,
        "WTD_opt": WTD_best,

        "IP_change_pct": (IP_best / float(base10[IDX["IP"]]) - 1.0) if base10[IDX["IP"]] != 0 else np.nan,
        "LAI_change_pct": (LAI_best / float(base10[IDX["LAI"]]) - 1.0) if base10[IDX["LAI"]] != 0 else np.nan,
        "WTD_change_pct": (WTD_best / float(base10[IDX["WTD"]]) - 1.0) if base10[IDX["WTD"]] != 0 else np.nan,

        "fixed_max_abs_diff": fixed_max_abs_diff,
        "pareto_size": int(ObjV.shape[0]),
        "row_seed": row_seed,
        "sample_weight": float(df.loc[i, "_sample_weight"]),
        "status": "ok",
    }
    summary_buffer.append(summary_row)

    # ---- 保存帕累托解集（长表）----
    pareto_part = []
    for j in range(ObjV.shape[0]):
        rr_j = float(np.clip(ObjV[j, 0], 0.0, 1.0))
        cost_j = float(np.maximum(ObjV[j, 1], 0.0))
        y_opt_j = float(np.clip(y_opt_all[j], 0.0, 1.0))

        ip_j, lai_j, wtd_j = float(Vars3[j,0]), float(Vars3[j,1]), float(Vars3[j,2])

        # 成本分项（复用上面已向量化好的结果）
        C_ISA_j = float(C_ISA_all[j, 0])
        C_WT_j = float(C_WT_all[j, 0])
        C_LAI_j = float(C_LAI_all[j, 0])
        C_total_j = float(C_total_all[j, 0])

        pareto_part.append({
            "row_id": i,
            "source_row_id": int(df.loc[i, "_source_row_id"]),
            "sol_id": j,
            "Longitude": float(df.loc[i, "Longitude"]),
            "Latitude": float(df.loc[i, "Latitude"]),
            "NAME_EN_JX": name_en_jx,
            "y_pred_base": float(np.clip(base_y, 0.0, 1.0)),
            "y_pred_opt": y_opt_j,
            "RR": rr_j,                       # 已强制 [0,1]
            "Cost_total": max(C_total_j, 0.0),  # 已强制 >=0
            "Cost_ISA": max(C_ISA_j, 0.0),
            "Cost_WT": max(C_WT_j, 0.0),
            "Cost_LAI": max(C_LAI_j, 0.0),

            "IP_opt": ip_j,
            "LAI_opt": lai_j,
            "WTD_opt": wtd_j,
            "row_seed": row_seed,
            "sample_weight": float(df.loc[i, "_sample_weight"]),
        })

    pareto_buffer.extend(pareto_part)

    # ---- 周期性落盘 ----
    if (len(summary_buffer) >= int(SAVE_EVERY_ROWS)) or (run_k == len(todo_indices)):
        # 先写 pareto，再写 summary；summary 被当作“该 row 完成”的标记
        if len(pareto_buffer) > 0:
            _append_df_safely(pd.DataFrame(pareto_buffer), CHECKPOINT_PARETO_PATH)
            pareto_buffer = []

        if len(summary_buffer) > 0:
            _append_df_safely(pd.DataFrame(summary_buffer), CHECKPOINT_SUMMARY_PATH)
            summary_buffer = []

        gc.collect()

    if (run_k % 10 == 0) or (run_k == len(todo_indices)):
        print(
            f"[{run_k}/{len(todo_indices)}] row_id={i} "
            f"base={base_y:.6f} y_opt={y_opt_best:.6f} RR_best={best_RR:.4f} "
            f"Cost_best={best_Cost:.3e} pareto={ObjV.shape[0]}"
        )

def _scale_costs_for_export(df_in: pd.DataFrame) -> pd.DataFrame:
    """导出时不再做任何成本缩放，保持单点优化得到的原始成本值。"""
    return df_in.copy()

summary_df = _read_ckpt_csv(CHECKPOINT_SUMMARY_PATH)
pareto_df = _read_ckpt_csv(CHECKPOINT_PARETO_PATH)

if "row_id" in summary_df.columns:
    summary_df = summary_df.drop_duplicates(subset=["row_id"], keep="last").sort_values("row_id").reset_index(drop=True)

if ("row_id" in pareto_df.columns) and ("sol_id" in pareto_df.columns):
    pareto_df = pareto_df.drop_duplicates(subset=["row_id", "sol_id"], keep="last").sort_values(["row_id", "sol_id"]).reset_index(drop=True)

summary_df = _scale_costs_for_export(summary_df)
pareto_df = _scale_costs_for_export(pareto_df)

summary_df.to_csv(FINAL_SUMMARY_PATH, index=False, encoding="utf-8-sig")
pareto_df.to_csv(FINAL_PARETO_PATH, index=False, encoding="utf-8-sig")

print("summary_df:", summary_df.shape)
print("pareto_df :", pareto_df.shape)

display(summary_df.head())

print("\nCost summary (export values, no scaling):")
summary_cols = [c for c in ["Cost_best", "Cost_ISA", "Cost_WT", "Cost_LAI", "Cost_total_check"] if c in summary_df.columns]
if summary_cols:
    display(summary_df[summary_cols].describe(percentiles=[0.5, 0.9, 0.99]).T)
